# Tile QC — EMIT Cloud Mask

Runs after `Color_Matching.ipynb`. Downloads EMIT L2A mask NCs, reprojects
cloud/cirrus flags to the same UTM grid, flags tiles by cloud fraction + S2 brightness.
All thresholds come from `pipeline_config.yaml`.

In [ ]:
import os, textwrap
from google.colab import userdata

os.environ["GH_USER"] = userdata.get("GH_USER")
os.environ["GH_TOKEN"] = userdata.get("GH_TOKEN")

askpass_path = "/tmp/gh_askpass.sh"
with open(askpass_path, "w") as f:
    f.write(textwrap.dedent("""\
        #!/bin/sh
        case "$1" in
          *Username*) echo "$GH_USER" ;;
          *Password*) echo "$GH_TOKEN" ;;
          *) echo "" ;;
        esac
    """))
os.chmod(askpass_path, 0o700)
os.environ["GIT_ASKPASS"] = askpass_path
os.environ["GIT_TERMINAL_PROMPT"] = "0"

!git clone https://github.com/incredet/HyperSpectralSuperResolution.git
%cd /content/HyperSpectralSuperResolution/

In [ ]:
!pip install -q numpy scipy rasterio matplotlib tqdm earthaccess netCDF4 pyproj shapely xarray python-dateutil pyyaml

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import earthaccess as ea
ea.login(strategy="environment")

In [ ]:
DRIVE_BASE = "/content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches/2026-03-22"

!python data/tile_qc.py --drive-base "$DRIVE_BASE"

## Diagnostics — distributions & threshold sweep

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml

drive_base = Path(DRIVE_BASE)
with open(drive_base / "pipeline_config.yaml") as f:
    cfg = yaml.safe_load(f)

max_cloud = cfg["qc_max_emit_cloud_frac"]
max_bright = cfg["qc_max_s2_bright_frac"]

qc = pd.read_csv(drive_base / "r2_tiles_qc.csv")
print(f"Total tiles: {len(qc)}")

# ── 1. Marginal distributions ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].hist(qc["combined_frac"].dropna(), bins=60, edgecolor="black", alpha=0.7)
axes[0].axvline(max_cloud, color="red", ls="--", label=f"thresh={max_cloud}")
axes[0].set(xlabel="EMIT cloud+cirrus frac", ylabel="Count")
axes[0].legend()

axes[1].hist(qc["s2_bright_frac"].dropna(), bins=60, edgecolor="black", alpha=0.7)
axes[1].axvline(max_bright, color="red", ls="--", label=f"thresh={max_bright}")
axes[1].set(xlabel="S2 bright pixel frac", ylabel="Count")
axes[1].legend()

axes[2].hist(qc["r2_mean"].dropna(), bins=60, edgecolor="black", alpha=0.7)
axes[2].set(xlabel="Mean R²", ylabel="Count")

plt.tight_layout()
plt.show()

# ── 2. Joint scatter plots ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].scatter(qc["r2_mean"], qc["combined_frac"], alpha=0.15, s=4, c="steelblue")
axes[0].axhline(max_cloud, color="red", ls="--", lw=0.8)
axes[0].set(xlabel="Mean R²", ylabel="Cloud+cirrus frac")

axes[1].scatter(qc["r2_mean"], qc["s2_bright_frac"], alpha=0.15, s=4, c="darkorange")
axes[1].axhline(max_bright, color="red", ls="--", lw=0.8)
axes[1].set(xlabel="Mean R²", ylabel="S2 bright frac")

axes[2].scatter(qc["combined_frac"], qc["s2_bright_frac"], alpha=0.15, s=4, c="seagreen")
axes[2].axhline(max_bright, color="red", ls="--", lw=0.8)
axes[2].axvline(max_cloud, color="red", ls="--", lw=0.8)
axes[2].set(xlabel="Cloud+cirrus frac", ylabel="S2 bright frac")

plt.tight_layout()
plt.show()

# ── 3. Conditional: cloud fraction grouped by R² bins ────────────────
qc["r2_bin"] = pd.cut(qc["r2_mean"], bins=[0, 0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 1.0])
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
qc.boxplot(column="combined_frac", by="r2_bin", ax=axes[0], rot=45)
axes[0].set(title="Cloud frac by R² bin", xlabel="R² bin", ylabel="Cloud+cirrus frac")
axes[0].get_figure().suptitle("")

qc.boxplot(column="s2_bright_frac", by="r2_bin", ax=axes[1], rot=45)
axes[1].set(title="S2 bright frac by R² bin", xlabel="R² bin", ylabel="S2 bright frac")
axes[1].get_figure().suptitle("")

plt.tight_layout()
plt.show()

# ── 4. Threshold sweep — how many tiles survive each combo ───────────
r2_cuts = [0.0, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85]
cloud_cuts = [1.0, 0.20, 0.10, 0.05, 0.02, 0.01, 0.0]
bright_cuts = [1.0, 0.20, 0.10, 0.05, 0.02, 0.0]

print("\n── Threshold sweep (R² × cloud × bright) ──")
print(f"{'R²≥':>6} {'cloud≤':>8} {'bright≤':>9} {'pass':>7} {'%':>7}")
print("-" * 42)
for r2t in r2_cuts:
    for ct in cloud_cuts:
        for bt in bright_cuts:
            mask = (qc["r2_mean"] >= r2t) & (qc["combined_frac"] <= ct) & (qc["s2_bright_frac"] <= bt)
            n = mask.sum()
            if n == 0:
                continue
            print(f"{r2t:>6.2f} {ct:>8.2f} {bt:>9.2f} {n:>7d} {100*n/len(qc):>6.1f}%")

# ── 5. Intersection sizes for key combos ─────────────────────────────
print("\n── Key intersections ──")
for r2t in [0.70, 0.75, 0.80]:
    r2_ok = qc["r2_mean"] >= r2t
    cloud_ok = qc["combined_frac"] <= max_cloud
    bright_ok = qc["s2_bright_frac"] <= max_bright

    n_r2 = r2_ok.sum()
    n_cloud = (r2_ok & cloud_ok).sum()
    n_bright = (r2_ok & bright_ok).sum()
    n_all = (r2_ok & cloud_ok & bright_ok).sum()

    print(f"\nR² >= {r2t}:")
    print(f"  R² only:              {n_r2:>6d}")
    print(f"  + cloud ≤ {max_cloud}:       {n_cloud:>6d}  (−{n_r2-n_cloud})")
    print(f"  + bright ≤ {max_bright}:      {n_bright:>6d}  (−{n_r2-n_bright})")
    print(f"  all three:            {n_all:>6d}  (−{n_r2-n_all} from R² set)")

fig.savefig(str(drive_base / "qc_diagnostics.png"), dpi=150, bbox_inches="tight")
print(f"\nSaved qc_diagnostics.png")